In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.amp import autocast, GradScaler

from PIL import Image
from PIL import ImageFile

import pandas
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import os

from tqdm import tqdm
import copy

In [2]:
class FilteredImageFolder(ImageFolder):
    def __init__(self, root, excluded_classes=None, **kwargs):
        self.excluded_classes = set(excluded_classes) if excluded_classes else set()
        super().__init__(root, **kwargs)

    def find_classes(self, directory):
        # Let the parent find all classes first
        classes, class_to_idx = super().find_classes(directory)
        # Filter out the unwanted classes
        if self.excluded_classes:
            classes = [c for c in classes if c not in self.excluded_classes]
        # Re-build the dictionary to ensure indices are contiguous (0, 1, 2...)
        # If we didn't do this, removing the middle folder might result in indices [0, 2]
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

        return classes, class_to_idx

In [3]:
class DualStreamTransform:
    def __init__(self, patch_size=256, rgb_size=768): #rgb_size can be ignored since manually set the size to 256
        self.patch_size = patch_size
        self.rgb_size = rgb_size
        self.hann_window = self._make_hann_window(patch_size)

        # 1. RGB Stream Transforms (Standard ImageNet normalization)
        self.rgb_transform = transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop((256,256)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.4932, 0.4570, 0.4053], std=[0.3003, 0.2854, 0.2941]) #ImageNet: [0.485, 0.456, 0.406] Calculated: [0.4932, 0.4570, 0.4053]
                                                                                              #ImageNet: [0.229, 0.224, 0.225] Calculated: [0.3003, 0.2854, 0.2941]
        ])

        # Laplacian kernel
        lap = torch.tensor([[0, 1, 0], [1,-4, 1], [0, 1, 0]], dtype=torch.float32)
        self.lap_kernel = lap.unsqueeze(0).unsqueeze(0)

    def _make_hann_window(self, ps):
        hann1d = torch.hann_window(ps)
        return torch.outer(hann1d, hann1d)

    def _laplacian_hpf(self, patch):
        return torch.nn.functional.conv2d(patch, self.lap_kernel, padding=1)[0,0]

    def _fft_map(self, img_2d):
        f = torch.fft.fft2(img_2d)
        f = torch.fft.fftshift(f)
        mag = torch.abs(f)
        log_mag = torch.log(mag + 1e-6)
        return (log_mag - log_mag.mean()) / (log_mag.std() + 1e-8)

    def _pad_and_patch(self, gray):
        # Resize to ensuring splitting is clean (768 by 768)
        gray = TF.resize(gray.unsqueeze(0), [self.rgb_size, self.rgb_size])[0]
        
        ps = self.patch_size
        patches = []
        # Extract patches
        for y in range(0, self.rgb_size - ps + 1, ps):
            for x in range(0, self.rgb_size - ps + 1, ps):
                patches.append(gray[y:y+ps, x:x+ps])
        return patches

    def __call__(self, img):
        #Stream 1: RGB
        rgb_tensor = self.rgb_transform(img)

        #Stream 2: FFT Patches
        # Convert to grayscale
        gray = TF.to_tensor(img).mean(dim=0) 
        
        # Get raw patches
        raw_patches = self._pad_and_patch(gray)
        
        processed_patches = []
        for p in raw_patches:
            p = p.unsqueeze(0) # [1, H, W]
            hpf = self._laplacian_hpf(p) * self.hann_window
            fft = self._fft_map(hpf)
            processed_patches.append(fft.unsqueeze(0)) # [1, H, W]

        # Stack into one tensor [Num_Patches, 1, H, W]
        patch_tensor = torch.stack(processed_patches)

        return rgb_tensor, patch_tensor

class DualStreamDataset(Dataset):
    def __init__(self, data_dir, transform):
        self.base_ds = FilteredImageFolder(data_dir,transform=None,excluded_classes=["RealArt"]) #Changed to filtered to match dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_ds)

    def __getitem__(self, idx):
        img, label = self.base_ds[idx]
        
        # Apply the DualStreamTransform
        # returns (rgb, patches)
        rgb, patches = self.transform(img) 
        
        return (rgb, patches), label

    @property
    def classes(self):
        return self.base_ds.classes

In [ ]:
class DualStreamNetwork(nn.Module):
    def __init__(self, num_classes):
        super(DualStreamNetwork, self).__init__()

        #Stream 1: RGB Model
        self.rgb_backbone = torchvision.models.efficientnet_b0()
        self.rgb_backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_features=1280, out_features=2)
        )
        # self.rgb_backbone.load_state_dict(torch.load('/kaggle/input/centad_cnn-v1.1/pytorch/default/14/centad_v1.10.pth')) #GPU
        # self.rgb_backbone.load_state_dict(torch.load('/kaggle/input/centad_cnn-v1.1/pytorch/default/13/centad_v1.9.pth',map_location=torch.device('cpu'))) #!Update model path here
        # Remove the last FC layer to get features instead of predictions
        self.rgb_feature_dim = self.rgb_backbone.classifier[1].in_features #1280
        self.rgb_backbone.classifier = nn.Identity()
        for param in self.rgb_backbone.parameters(): #Freeze all parameters
            param.requires_grad = False
        
        #Stream 2: FFT Model
        self.fft_backbone = models.efficientnet_b0() #weights=torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1
        
        # Modify first conv layer: 3 channels -> 1 channel
        first_conv_layer = self.fft_backbone.features[0][0]
        self.fft_backbone.features[0][0] = nn.Conv2d(
            in_channels=1, 
            out_channels=first_conv_layer.out_channels, 
            kernel_size=first_conv_layer.kernel_size, 
            stride=first_conv_layer.stride, 
            padding=first_conv_layer.padding, 
            bias=False
        )
        
        self.fft_feature_dim = self.fft_backbone.classifier[1].in_features
        self.fft_backbone.classifier = nn.Identity() # Remove classifier

        #Fusion
        #concat RGB features + FFT features
        self.fusion_dim = self.rgb_feature_dim + self.fft_feature_dim
        
        self.classifier = nn.Sequential(
            nn.Linear(self.fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, rgb, patches):
        # 1. Process RGB Stream
        # rgb shape: [Batch, 3, 1024, 1024]
        rgb_feat = self.rgb_backbone(rgb) # [Batch, rgb_dim]

        # 2. Process FFT Patch Stream
        # patches shape: [Batch, 16, 1, 256, 256]
        B, NumPatches, C, H, W = patches.shape
        
        # Combine Batch and Patches dimensions to feed into CNN
        # New shape: [Batch * 16, 1, 256, 256]
        patches_flat = patches.view(B * NumPatches, C, H, W)
        
        fft_feat = self.fft_backbone(patches_flat) # [Batch*16, fft_dim]
        
        # Reshape back to separate Batch and Patches
        fft_feat = fft_feat.view(B, NumPatches, -1) # [Batch, 16, fft_dim]
        
        # Aggregate features
        fft_feat = fft_feat.mean(dim=1) # [Batch, fft_dim]

        # 3. Concatenate
        combined = torch.cat((rgb_feat, fft_feat), dim=1)

        # 4. Classify
        out = self.classifier(combined)
        return out

In [ ]:
#Training configuration
device = torch.device('cuda')

dual_transform = DualStreamTransform() #patch_size=256

train_set = DualStreamDataset(data_dir="/kaggle/input/diffusion-image-detection/Set/TrainSet", transform=dual_transform)
test_set = DualStreamDataset(data_dir="/kaggle/input/training-set/test/test", transform=dual_transform)
val_set = DualStreamDataset(data_dir="/kaggle/input/genimagesmalltest/Test", transform=dual_transform)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True) # Reduced batch size due to memory usage
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)
val_loader = DataLoader(val_set,batch_size=16,shuffle=False)

model = DualStreamNetwork(num_classes=2).to(device)

lr = 0.0005
n_epochs = 4

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr,weight_decay = 0.0001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max = n_epochs)
scaler = GradScaler()

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

In [ ]:
#Training Loop
best_val_loss = float('inf')

patience = 3
epochs_no_improve = 0
epochs_cycled = 0

best_model_weights = copy.deepcopy(model.state_dict())

for epoch in tqdm(range(n_epochs)):
    epochs_cycled += 1
    print(f"Epoch {epoch+1}/{n_epochs}")
    print("-" * 10)

    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    
    # Wrap loader with tqdm for progress bar
    train_loop = tqdm(train_loader, desc="Training",position=0, leave=True)
    
    for i, (inputs, labels) in enumerate(train_loop):
        # Unpack tuple from Dataset
        rgb, patches = inputs
        
        rgb = rgb.to(device)
        patches = patches.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Mixed Precision Training
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(rgb, patches)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        
        # Update progress bar description or not since it lags the terminal
        # train_loop.set_postfix(loss=loss.item())

    avg_train_loss = running_loss / len(train_loader)
    print(f"Avg Train Loss: {avg_train_loss:.4f}")
    
    # Step scheduler usually after epoch
    scheduler.step()

    #Validation
    model.eval() # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0
    
    # No gradient calculation needed for validation (saves memory/time)
    with torch.no_grad():
        val_loop = tqdm(test_loader, desc="Validating",position=0, leave=True)
        
        for inputs, labels in val_loop:
            rgb, patches = inputs
            rgb = rgb.to(device)
            patches = patches.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(rgb, patches)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            # Calculate Accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(test_loader)
    accuracy = 100 * correct / total

    print(f"Validation Loss: {avg_val_loss:.4f} | Accuracy: {accuracy:.2f}%")
    epochs_cycled += 1
    #Checkpoint system, saves model state every epoch.
    torch.save({
        'epoch': epochs_cycled,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
    }, f"/kaggle/working/checkpoint_epoch_{epochs_cycled}.pth")
    
    # --- SAVE BEST MODEL ---
    if avg_val_loss < best_val_loss:
        print(f"Validation Loss Improved ({best_val_loss:.4f} -> {avg_val_loss:.4f}).")
        best_model_weights = copy.deepcopy(model.state_dict())
        best_val_loss = avg_val_loss
        # torch.save(model.state_dict(), "best_dual_stream_model.pth")
        epochs_no_improve = 0
    else:
        print("No improvement")
        epochs_no_improve += 1
    if epochs_no_improve > patience:
        print(f"No improvement after {patience} epochs, stopping training.")
        break
    print("\n") # New line for next epoch
    

In [ ]:
# save_folder = os.listdir('/kaggle/input/centad_cnn-v1.1/pytorch/default/12') #Update this please.
# folder_length = len(save_folder) + 1

PATH = f'/kaggle/working/DualStreamV1.2.pth'
model.load_state_dict(best_model_weights)
torch.save(model.state_dict(), PATH)

In [ ]:
# #Display model loss throughout training
# epochs = range(1, epochs_cycled+1)

# #plt.figure(figsize=(10, 6)) # Optional: set figure size for better visualization
# fig,ax = plt.subplots()
# plt.plot(epochs, train_loss_arr, label='Training Loss', color='blue')
# plt.plot(epochs, val_loss_arr, label='Validation Loss', color='red')


# plt.title('Training and Validation Loss Curve')
# plt.xlabel('Epochs')
# plt.ylabel('Loss')
# ax.xaxis.set_major_locator(MaxNLocator(integer=True))
# plt.legend() # Display the labels for each curve
# plt.grid(True) # Optional: add a grid for readability
# plt.show()

In [ ]:
val_set = DualStreamDataset(data_dir="Insert dataset path here", transform=dual_transform)
val_loader = DataLoader(val_set,batch_size=16,shuffle=False)

device = torch.device('cuda')
#Change this based on model version
PATH = '~/DualStreamV1.2.pth' #Insert path to model file.

loaded_model = DualStreamNetwork(num_classes=2).to(device)
loaded_model.load_state_dict(torch.load(PATH))
loaded_model.eval()
criterion = nn.CrossEntropyLoss()
val_loss = 0
with torch.no_grad():
    n_correct = 0
    n_samples = len(val_loader.dataset)
    for images, labels in val_loader:
        rgb, patches = images
        rgb = rgb.to(device)
        patches = patches.to(device)
        labels = labels.to(device)
        
        outputs = loaded_model(rgb,patches)
        loss = criterion(outputs,labels)
        val_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        n_correct += (predicted == labels).sum().item()
    acc = 100.0 * n_correct / n_samples
    print(f'Accuracy of the model: {acc} %\nModel loss: {val_loss/len(val_loader):.3f}')

In [ ]:
def show_patches(patch_tensor):
    # patch_tensor shape: [9, 1, 256, 256]
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    plt.subplots_adjust(left=0, right=1, bottom=0, top=0.95, wspace=0.05, hspace=0.025)
    fig.suptitle("FFT Patches", fontsize=12)
    axes = axes.flatten()
    for i in range(len(patch_tensor)):
        patch = patch_tensor[i].squeeze().cpu().numpy()
        axes[i].imshow(patch)
        axes[i].axis('off')
    plt.show()

In [ ]:
#Final product:
device = torch.device('cpu')
PATH = '/kaggle/input/dualstreamcentad/pytorch/default/3/DualStreamV1.2.pth' #Comment if training

dual_transform = DualStreamTransform()

loaded_model = DualStreamNetwork(num_classes=2).to(device)
loaded_model.load_state_dict(torch.load(PATH,map_location=torch.device('cpu')))
loaded_model.eval()
# loaded_model.to(device)
# target_layers = [loaded_model.features[7]] #EfficientNetb0
# target_layers = [loaded_model.layer4[-1]] #ResNet50

img_path = '/kaggle/input/aitest/REAL_MalaPlate.jpg' #Change this path to any image's path
image = Image.open(img_path).convert("RGB")  # ensure 3 channels
plt.imshow(image)
plt.axis('off')
plt.title("RGB Image")
plt.show()
rgb_tensor, patch_tensor = dual_transform(image)
show_patches(patch_tensor)
rgb_input = rgb_tensor.unsqueeze(0).to(device)       # Shape: [1, 3, 256, 256]
patch_input = patch_tensor.unsqueeze(0).to(device)   # Shape: [1, 9, 1, 256, 256]

print(f"RGB Shape: {rgb_input.shape}")
print(f"Patch Shape: {patch_input.shape}")

#print(train_dataset.classes)
test_class = ['fake', 'real']

with torch.no_grad():
    output = loaded_model(rgb_input,patch_input)
    #print(output)
    probs = torch.softmax(output, dim=1)
    probs = probs.cpu().numpy().tolist()
    #print(probs)
    _, predicted = torch.max(output, 1)
    print(f"Predicted index: {predicted.item()}\nPredicted class: {test_class[predicted.item()]}")
    print(f"---Probabilities---\nFake: {probs[0][0]*100:.3f}%\nReal: {probs[0][1]*100:.3f}%")